# Metal--ligand recognition prototype

This notebook illustrates the analyzer-level metal--ligand candidate records introduced for coordination chemistry. The key separation is:

- `ML/sigma-acceptor`: ligand-to-metal sigma donation into a metal acceptor channel.
- `ML/pi-donor`: metal-to-ligand pi back-donation, where the metal acts as the pi donor and the ligand acts as a pi acceptor.

The records are analyzer diagnostics. They are not automatic NBO Lewis assignments and they are not automatic VB active-space choices.

In [23]:
from types import SimpleNamespace
import importlib
from pathlib import Path
import sys
import numpy as np

# Prefer the source tree when the notebook is run from the repository or docs/.
def find_repo_root(start):
    for path in (start, *start.parents):
        if (path / 'src' / 'pymodule').exists():
            return path
    return start

def load_source_module(module_name, relative_file):
    sys.modules.pop(module_name, None)
    module_path = repo / 'src' / 'pymodule' / relative_file
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

repo = find_repo_root(Path.cwd().resolve())
import veloxchem
source_pkg = str((repo / 'src' / 'pymodule').resolve())
if source_pkg not in veloxchem.__path__:
    veloxchem.__path__.insert(0, source_pkg)
sys.modules.pop('veloxchem.orbitalanalyzerdriver', None)

from veloxchem.molecule import Molecule
from veloxchem.orbitalanalyzerdriver import _build_orbital_candidates

nbodriver_module = load_source_module('veloxchem.nbodriver', 'nbodriver.py')
vbdriver_module = load_source_module('veloxchem.vbdriver', 'vbdriver.py')

## Mock diagnostic payloads for fast development

The first examples below use small mock NAO density blocks. They are useful for fast regression of the recognition logic, but they are **not real quantum-chemical calculations**. The real SCF workflow is shown in the final section.

In [17]:
def pd_ligand_molecule(ligand):
    if ligand == 'NH3':
        xyz = '''
Pd  0.0000  0.0000  0.0000
N   2.0500  0.0000  0.0000
H   2.4500  0.9300  0.0000
H   2.4500 -0.4650  0.8050
H   2.4500 -0.4650 -0.8050
'''
    elif ligand == 'PH3':
        xyz = '''
Pd  0.0000  0.0000  0.0000
P   2.2500  0.0000  0.0000
H   2.8500  1.1500  0.0000
H   2.8500 -0.5750  0.9960
H   2.8500 -0.5750 -0.9960
'''
    elif ligand == 'carbene':
        xyz = '''
Pd  0.0000  0.0000  0.0000
C   2.0000  0.0000  0.0000
N   2.6500  1.0500  0.0000
N   2.6500 -1.0500  0.0000
H   3.4500  1.2000  0.0000
H   3.4500 -1.2000  0.0000
'''
    else:
        raise ValueError(ligand)
    mol = Molecule.read_str(xyz)
    mol.set_charge(0)
    mol.set_multiplicity(1)
    return mol

def mock_nao_data(ligand, sigma_coupling, pi_coupling):
    # NAO order: Pd d, Pd d, Pd d, Pd s/p acceptor, ligand core, ligand sigma donor,
    # ligand pi acceptor p, ligand pi acceptor d/p.
    atom_map = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=int)
    angular_map = np.array([2, 2, 2, 0, 0, 1, 1, 2], dtype=int)
    populations = np.array([1.75, 1.70, 1.65, 0.25, 2.00, 1.90, 0.20, 0.20])
    density = np.diag(populations)
    density[5, 3] = density[3, 5] = sigma_coupling
    density[0, 7] = density[7, 0] = pi_coupling
    density[1, 7] = density[7, 1] = 0.5 * pi_coupling
    density[2, 6] = density[6, 2] = 0.25 * pi_coupling
    return SimpleNamespace(
        populations=populations,
        density=density,
        atom_map=atom_map,
        angular_momentum_map=angular_map,
    )

def analyzer_payload(ligand, sigma_coupling, pi_coupling):
    molecule = pd_ligand_molecule(ligand)
    nao_data = mock_nao_data(ligand, sigma_coupling, pi_coupling)
    candidates = _build_orbital_candidates(
        molecule,
        nao_data,
        include_metal_ligand_candidates=True,
    )
    n_nao = len(nao_data.populations)
    nao_data.transform = np.eye(n_nao)
    nao_data.overlap = np.eye(n_nao)
    spin_data = SimpleNamespace(
        alpha_density=0.5 * nao_data.density,
        beta_density=0.5 * nao_data.density,
        spin_density=np.zeros_like(nao_data.density),
        spin_populations=np.zeros_like(nao_data.populations),
        unpaired_electrons=0.0,
    )
    analysis = SimpleNamespace(
        overlap=np.eye(n_nao),
        density=nao_data.density,
        nao_data=nao_data,
        spin_data=spin_data,
        mo_analysis={},
        orbital_candidates=candidates,
    )
    return molecule, nao_data, analysis

def metal_ligand_records(ligand, sigma_coupling, pi_coupling):
    _, _, analysis = analyzer_payload(ligand, sigma_coupling, pi_coupling)
    return [c for c in analysis.orbital_candidates if c.get('type') == 'ML']

def summarize(records):
    rows = []
    for rec in records:
        rows.append({
            'subtype': rec['subtype'],
            'channel': rec['channel'],
            'role': rec.get('donor_acceptor_role'),
            'donation': round(float(rec.get('donation_strength', 0.0)), 4),
            'back_donation': round(float(rec.get('back_donation_strength', 0.0)), 4),
        })
    return rows

def compute_nbo_vb(ligand, sigma_coupling, pi_coupling):
    molecule, nao_data, analysis = analyzer_payload(ligand, sigma_coupling, pi_coupling)

    class MockOrbitalAnalyzer:
        def __init__(self, *args, **kwargs):
            pass
        def run(self):
            return analysis
        def classify_nao_data(self, _nao_data):
            return analysis

    original_nbo_analyzer = nbodriver_module.OrbitalAnalyzer
    original_vb_analyzer = vbdriver_module.OrbitalAnalyzer
    nbodriver_module.OrbitalAnalyzer = MockOrbitalAnalyzer
    vbdriver_module.OrbitalAnalyzer = MockOrbitalAnalyzer
    try:
        nbo_driver = nbodriver_module.NboDriver()
        nbo_driver.verbose = False
        nbo = nbo_driver.compute(
            molecule,
            basis=SimpleNamespace(),
            mol_orbs=SimpleNamespace(),
            options=nbodriver_module.NboComputeOptions(
                include_diagnostics=False,
                include_mo_analysis=False,
                max_alternatives=0,
            ),
        )
        vb = vbdriver_module.VbDriver().compute(
            molecule,
            basis=SimpleNamespace(get_dimensions_of_basis=lambda: len(nao_data.populations)),
            options=vbdriver_module.VbComputeOptions(use_active_space=False),
            nao_data=nao_data,
        )
    finally:
        nbodriver_module.OrbitalAnalyzer = original_nbo_analyzer
        vbdriver_module.OrbitalAnalyzer = original_vb_analyzer
    return nbo, vb

def summarize_compute(nbo, vb):
    return {
        'NBO.compute': summarize(nbo['metal_ligand_diagnostics']),
        'VB.compute': summarize(vb['diagnostics']['candidate_partitions']['metal_ligand']),
        'NBO primary has ML': any(c.get('type') == 'ML' for c in nbo['nbo_list']),
    }

## Pd--NH3 versus Pd--PH3

The mock payload below encodes a stronger metal-to-ligand pi back-donation block for phosphine than for ammine. The analyzer should keep sigma donation and pi back-donation as separate records.

In [3]:
examples = {
    'Pd--NH3': metal_ligand_records('NH3', sigma_coupling=0.12, pi_coupling=0.04),
    'Pd--PH3': metal_ligand_records('PH3', sigma_coupling=0.13, pi_coupling=0.20),
}

for name, records in examples.items():
    print(f'\n{name}')
    for row in summarize(records):
        print(row)


Pd--NH3
{'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.12, 'back_donation': 0.0}
{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi acceptor', 'donation': 0.0, 'back_donation': 0.0458}

Pd--PH3
{'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.13, 'back_donation': 0.0}
{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi acceptor', 'donation': 0.0, 'back_donation': 0.2291}


In [4]:
nh3_pi = next(r for r in examples['Pd--NH3'] if r['subtype'] == 'pi-donor')
ph3_pi = next(r for r in examples['Pd--PH3'] if r['subtype'] == 'pi-donor')
print('Pd--PH3 back donation > Pd--NH3 back donation:', ph3_pi['back_donation_strength'] > nh3_pi['back_donation_strength'])
print('NH3:', nh3_pi['back_donation_strength'])
print('PH3:', ph3_pi['back_donation_strength'])

Pd--PH3 back donation > Pd--NH3 back donation: True
NH3: 0.0458257569495584
PH3: 0.22912878474779202


## Pd--carbene illustration

A carbene ligand can be represented as a strong sigma donor with a pi-acceptor channel. The analyzer still reports only neutral candidate diagnostics; deciding whether this becomes an NBO resonance model or a VB active space remains the responsibility of `NboDriver` or `VbDriver`.

In [5]:
carbene_records = metal_ligand_records('carbene', sigma_coupling=0.22, pi_coupling=0.16)
for row in summarize(carbene_records):
    print(row)

{'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.22, 'back_donation': 0.0}
{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi acceptor', 'donation': 0.0, 'back_donation': 0.1833}


## Mock NBO and VB compute workflow

This compact cell runs the mock Pd--NH3 and Pd--PH3 analyzer payloads through `NboDriver.compute()` and `VbDriver.compute()`. This checks the driver result dictionaries, but it still uses mock NAO data rather than an SCF density.

In [18]:
for name, args in {
    'Pd--NH3': ('NH3', 0.12, 0.04),
    'Pd--PH3': ('PH3', 0.13, 0.20),
}.items():
    nbo, vb = compute_nbo_vb(*args)
    print(f'\n{name}')
    print(summarize_compute(nbo, vb))


Pd--NH3
{'NBO.compute': [{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.0458}, {'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.12, 'back_donation': 0.0}], 'VB.compute': [{'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.12, 'back_donation': 0.0}, {'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.0458}], 'NBO primary has ML': False}

Pd--PH3
{'NBO.compute': [{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.2291}, {'subtype': 'sigma-acceptor', 'channel': 'ligand-to

The expected `NBO primary has ML` value is `False`: the compute workflow exposes metal--ligand records as diagnostics while keeping them out of the occupied Lewis assignment. The next section repeats the same idea with a real SCF density.

## Real SCF Pd--NH3 and Pd--PH3 workflow

This section performs the actual electronic-structure workflow: build a basis, run SCF, then pass the SCF molecular orbitals to `NboDriver.compute()`. The VB diagnostic view reuses the real NAO payload from the NBO/orbital-analysis result. If the current prototype does not identify a pi-back-donation channel from the real density, the notebook reports that explicitly rather than forcing the mock trend.

In [19]:
def run_real_scf_nbo_vb(ligand, basis_label='def2-svp'):
    molecule = pd_ligand_molecule(ligand)
    basis = veloxchem.MolecularBasis.read(molecule, basis_label, ostream=None)

    scf = veloxchem.ScfRestrictedDriver()
    scf.ostream.mute()
    scf.xcfun = 'hf'
    scf.compute(molecule, basis)

    nbo_driver = nbodriver_module.NboDriver()
    nbo_driver.verbose = False
    nbo_driver.ostream.mute()
    nbo = nbo_driver.compute(
        molecule,
        basis,
        scf.mol_orbs,
        options=nbodriver_module.NboComputeOptions(
            include_diagnostics=True,
            include_mo_analysis=False,
            max_alternatives=0,
        ),
    )

    vb = vbdriver_module.VbDriver().compute(
        molecule,
        basis,
        options=vbdriver_module.VbComputeOptions(use_active_space=False),
        nao_data=nbo['orbital_analysis'].nao_data,
    )
    return nbo, vb


def summarize_real_result(nbo, vb):
    return {
        'NBO.compute': summarize(nbo.get('metal_ligand_diagnostics', [])),
        'VB.compute': summarize(
            vb['diagnostics']['candidate_partitions'].get('metal_ligand', [])),
        'NBO primary has ML': any(c.get('type') == 'ML'
                                  for c in nbo.get('nbo_list', [])),
    }

In [20]:
real_results = {}
for label, ligand in {'Pd--NH3': 'NH3', 'Pd--PH3': 'PH3'}.items():
    nbo_real, vb_real = run_real_scf_nbo_vb(ligand, basis_label='def2-svp')
    real_results[label] = summarize_real_result(nbo_real, vb_real)
    print(f'\n{label}')
    print(real_results[label])

def first_pi_back(rows):
    for row in rows:
        if row['subtype'] == 'pi-donor':
            return row['back_donation']
    return None

nh3_back = first_pi_back(real_results['Pd--NH3']['NBO.compute'])
ph3_back = first_pi_back(real_results['Pd--PH3']['NBO.compute'])
if nh3_back is None or ph3_back is None:
    print('\nReal SCF note: the current prototype did not identify a pi-donor channel for both systems.')
else:
    print('\nReal SCF NBO back donation PH3 > NH3:', ph3_back > nh3_back)


Pd--NH3
{'NBO.compute': [{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.2432}, {'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.0, 'back_donation': 0.0}], 'VB.compute': [{'subtype': 'sigma-acceptor', 'channel': 'ligand-to-metal-sigma-donation', 'role': 'ligand sigma donor / metal acceptor', 'donation': 0.0, 'back_donation': 0.0}, {'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.2432}], 'NBO primary has ML': False}

Pd--PH3
{'NBO.compute': [{'subtype': 'pi-donor', 'channel': 'metal-to-ligand-pi-back-donation', 'role': 'metal pi donor / ligand pi nonbonding-or-acceptor', 'donation': 0.0, 'back_donation': 0.4935}, {'subtype': 'sigma-acceptor', 'channel': 'ligand-to-m

## VB-CI scan: sigma-only versus sigma + back-donation

This scan uses the real SCF density and the new combined metal--ligand active-space option in `VbDriver.compute()`. For each Pd--L distance, the sigma-donation pair is always active. The second calculation adds the pi back-donation pair, giving a 4-orbital/4-electron determinant-CI active space. The diagnostic quantity is

$\Delta E_{\pi} = E(\sigma + \pi) - E(\sigma\ \mathrm{only})$.

A negative value means that adding the back-donation channel lowers the fixed-orbital VB-CI energy in this model.

In [24]:
def pd_ligand_molecule_at_distance(ligand, distance):
    if ligand == 'NH3':
        donor, h_rel = 'N', [(0.40, 0.93, 0.00), (0.40, -0.465, 0.805), (0.40, -0.465, -0.805)]
    elif ligand == 'PH3':
        donor, h_rel = 'P', [(0.60, 1.15, 0.00), (0.60, -0.575, 0.996), (0.60, -0.575, -0.996)]
    else:
        raise ValueError(ligand)
    lines = [f'Pd 0.0000 0.0000 0.0000', f'{donor} {distance:.4f} 0.0000 0.0000']
    lines += [f'H {distance + dx:.4f} {dy:.4f} {dz:.4f}' for dx, dy, dz in h_rel]
    mol = Molecule.read_str('\n'.join(lines))
    mol.set_charge(0)
    mol.set_multiplicity(1)
    return mol


def vbci_sigma_pi_scan(ligand, distances, basis_label='def2-svp'):
    rows = []
    for distance in distances:
        molecule = pd_ligand_molecule_at_distance(ligand, distance)
        basis = veloxchem.MolecularBasis.read(molecule, basis_label, ostream=None)
        scf = veloxchem.ScfRestrictedDriver()
        scf.ostream.mute()
        scf.xcfun = 'hf'
        scf.compute(molecule, basis)

        nbo_driver = nbodriver_module.NboDriver()
        nbo_driver.verbose = False
        nbo_driver.ostream.mute()
        nbo = nbo_driver.compute(
            molecule,
            basis,
            scf.mol_orbs,
            options=nbodriver_module.NboComputeOptions(
                include_diagnostics=False,
                include_mo_analysis=False,
                max_alternatives=0,
            ),
        )

        def run_channels(channels):
            return vbdriver_module.VbDriver().compute(
                molecule,
                basis,
                options=vbdriver_module.VbComputeOptions(
                    mode='vbci',
                    optimize_orbitals=False,
                    active_metal_ligand_channels=channels,
                    freeze_inactive_orbitals=True,
                ),
                nao_data=nbo['orbital_analysis'].nao_data,
            )

        sigma = run_channels(('sigma-acceptor',))
        sigma_pi = run_channels(('sigma-acceptor', 'pi-donor'))
        e_sigma = float(sigma['energy'])
        e_sigma_pi = float(sigma_pi['energy'])
        row = {
            'R_PdL': distance,
            'E_sigma_only': round(e_sigma, 6),
            'E_sigma_plus_pi': round(e_sigma_pi, 6),
            'Delta_E_pi': round(e_sigma_pi - e_sigma, 6),
            'det_sigma': sigma['diagnostics']['determinant_count'],
            'det_sigma_pi': sigma_pi['diagnostics']['determinant_count'],
            'backdonation_strength': next(
                (round(x['back_donation'], 4) for x in summarize(nbo['metal_ligand_diagnostics'])
                 if x['subtype'] == 'pi-donor'),
                None,
            ),
        }
        rows.append(row)
    return rows


for ligand, distances in {'NH3': [1.95, 2.05, 2.15], 'PH3': [2.15, 2.25, 2.35]}.items():
    print(f'\nPd--{ligand}')
    for row in vbci_sigma_pi_scan(ligand, distances):
        print(row)


Pd--NH3
{'R_PdL': 1.95, 'E_sigma_only': -5.810442, 'E_sigma_plus_pi': -80.060392, 'Delta_E_pi': -74.249951, 'det_sigma': 4, 'det_sigma_pi': 36, 'backdonation_strength': 0.3017}
{'R_PdL': 2.05, 'E_sigma_only': -7.947141, 'E_sigma_plus_pi': -87.258704, 'Delta_E_pi': -79.311563, 'det_sigma': 4, 'det_sigma_pi': 36, 'backdonation_strength': 0.2432}
{'R_PdL': 2.15, 'E_sigma_only': 24.114645, 'E_sigma_plus_pi': -45.261167, 'Delta_E_pi': -69.375811, 'det_sigma': 4, 'det_sigma_pi': 36, 'backdonation_strength': 0.1593}

Pd--PH3
{'R_PdL': 2.15, 'E_sigma_only': 25.046812, 'E_sigma_plus_pi': -42.950497, 'Delta_E_pi': -67.99731, 'det_sigma': 4, 'det_sigma_pi': 36, 'backdonation_strength': 0.5739}
{'R_PdL': 2.25, 'E_sigma_only': 24.378924, 'E_sigma_plus_pi': -43.797362, 'Delta_E_pi': -68.176286, 'det_sigma': 4, 'det_sigma_pi': 36, 'backdonation_strength': 0.4935}
{'R_PdL': 2.35, 'E_sigma_only': 56.03424, 'E_sigma_plus_pi': -19.417401, 'Delta_E_pi': -75.451641, 'det_sigma': 4, 'det_sigma_pi': 36, 'ba

## Interpretation

The mock cells validate the intended sigma-donation/pi-back-donation plumbing. The real SCF section confirms that `NboDriver.compute()` and `VbDriver.compute()` expose a `ML/pi-donor` diagnostic when occupied metal d orbitals are compared with ligand pi-type nonbonding/acceptor space. With the current `def2-svp` examples, Pd--PH3 shows a larger real-density back-donation diagnostic than Pd--NH3, while `ML` records remain diagnostic-only in NBO.

The final scan now uses the intended active-space comparison: `E_sigma_only` is a 2-orbital/2-electron VB-CI model with only the sigma-donation channel active, and `E_sigma_plus_pi` is a 4-orbital/4-electron VB-CI model with sigma donation plus pi back-donation active. The reported `Delta_E_pi` is therefore the fixed-orbital VB-CI stabilization from switching on the back-donation pair within this prototype model.